# recs_014 — D2/D3 train + val head-to-head

Train learned rankers on frozen `two_tower_v1` pools, then compare on **external val** (no val tuning).

**Baselines in head-to-head:**
- `two_tower_v1` — pool retrieval order (no rerank)
- `two_tower_v1_heuristic_logpop_blend` — promoted D1 winner
- `popularity_train` — full-catalog popularity top-10
- `two_tower_v1_oracle` — best possible order within the frozen pool
- D2 / D3 models (when trained)

**Prereqs:**
```bash
python scripts/recs_job_build_example_cohort.py configs/recs_job_build_example_cohort_train_ranker.json
python scripts/recs_job_export_retrieval_pools.py configs/recs_job_export_retrieval_pools_train_ranker.json
python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json \
  --examples-parquet artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet
```

D1 reference: [`recs_013_ranker_d1_heuristic.ipynb`](recs_013_ranker_d1_heuristic.ipynb)

**Env:** D2 training uses TensorFlow — run in `tf_condaforge` (same as two-tower export).

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.example_cohort import load_retrieval_pool_rows, load_retrieval_pools_jsonl
from steam_review_ml.evaluation.heuristic_ranker import (
    METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND,
    pool_rerank_registry,
    rerank_scores_on_pool,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    average_precision_at_k,
    hit_rate_at_k,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
)

from steam_review_ml.recommender.ranker_d2_pointwise import (
    METHOD_TWO_TOWER_V1_RANKER_D2_POINTWISE_V1,
    PointwiseRankerConfig,
    load_pointwise_ranker,
    make_pool_score_fn as make_d2_pool_score_fn,
    pointwise_rows_from_pools,
    train_pointwise_ranker,
)
from steam_review_ml.recommender.ranker_d3_listwise import (
    METHOD_TWO_TOWER_V1_RANKER_D3_LISTWISE_V1,
    ListwiseRankerConfig,
    listwise_rows_from_pools,
    load_listwise_ranker,
    make_pool_score_fn as make_d3_pool_score_fn,
    train_listwise_ranker,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

POOL_METHOD = "two_tower_v1"
K_FINAL = 10
MIN_REVIEW_CHARS = 30
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
SPLIT_SEED = 2027
TUNE_FRAC = 0.10
TRAIN_D2 = True
TRAIN_D3 = True

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet"
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"
D2_OUTPUT_DIR = REPO_ROOT / "artifacts/recs/rankers/d2_pointwise_v1"
D2_MODEL_PATH = D2_OUTPUT_DIR / "pointwise_ranker.keras"
D3_OUTPUT_DIR = REPO_ROOT / "artifacts/recs/rankers/d3_listwise_v1"
D3_MODEL_PATH = D3_OUTPUT_DIR / "listwise_ranker.keras"

for p in (TRAIN_POOLS_PARQUET, VAL_JSONL):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required artifact: {p}")

print(f"TRAIN_POOLS={TRAIN_POOLS_PARQUET}")
print(f"VAL_JSONL={VAL_JSONL}")
print(f"D2_MODEL_PATH={D2_MODEL_PATH}")
print(f"D3_MODEL_PATH={D3_MODEL_PATH}")

TRAIN_POOLS=/home/ryanr/workspace/steam_recommendations/artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet
VAL_JSONL=/home/ryanr/workspace/steam_recommendations/artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl
D2_MODEL_PATH=/home/ryanr/workspace/steam_recommendations/artifacts/recs/rankers/d2_pointwise_v1/pointwise_ranker.keras
D3_MODEL_PATH=/home/ryanr/workspace/steam_recommendations/artifacts/recs/rankers/d3_listwise_v1/listwise_ranker.keras


/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load pools + catalog

In [2]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT,
    min_review_chars=MIN_REVIEW_CHARS,
    artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row

print(f"train pools: {len(train_pools):,}  val pools: {len(val_pools):,}  catalog apps: {len(app_ids):,}")

train pools: 51,691  val pools: 12,500  catalog apps: 315


## Train fit / tune split (90/10, stratified by slice)

Hold out 10% of **train** examples as `train_tune` for D2 checkpoint selection. External val (`val_pools`) is never used for training decisions — see **D2 — pointwise training** below for the full loop.

In [3]:
def stratified_ex_idx_split(
    pools: list[dict[str, Any]], *, tune_frac: float, seed: int
) -> tuple[set[int], set[int]]:
    """Split ``ex_idx`` into fit/tune sets, stratified by ``slice_name`` (D2 early stop)."""
    rng = np.random.default_rng(seed)
    by_slice: dict[str, list[int]] = {}
    for row in pools:
        by_slice.setdefault(str(row["slice_name"]), []).append(int(row["ex_idx"]))
    fit_ids: set[int] = set()
    tune_ids: set[int] = set()
    for _slice, ids in by_slice.items():
        ids_arr = np.asarray(sorted(ids))
        rng.shuffle(ids_arr)
        n_tune = max(1, int(round(len(ids_arr) * tune_frac)))
        tune_ids.update(int(x) for x in ids_arr[:n_tune])
        fit_ids.update(int(x) for x in ids_arr[n_tune:])
    return fit_ids, tune_ids


fit_ex_idx, tune_ex_idx = stratified_ex_idx_split(train_pools, tune_frac=TUNE_FRAC, seed=SPLIT_SEED)
train_fit = [r for r in train_pools if int(r["ex_idx"]) in fit_ex_idx]
train_tune = [r for r in train_pools if int(r["ex_idx"]) in tune_ex_idx]
print(f"fit={len(train_fit):,}  tune={len(train_tune):,}")

fit=46,522  tune=5,169


## Shared helpers (pool → top-K metrics)

In [4]:
def pool_scores_to_ranked_indices(
    pool_app_ids: list[int],
    pool_scores: np.ndarray,
    *,
    k_final: int,
) -> np.ndarray:
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:k_final]


def popularity_catalog_ranked_indices(*, query_app_id: int, k_final: int) -> np.ndarray:
    """Full-catalog popularity top-k; masks ``query_app_id`` (``popularity_train`` baseline)."""
    s = np.asarray(pop_row, dtype=np.float64).copy()
    row = app_to_row.get(int(query_app_id))
    if row is not None:
        s[row] = -np.inf
    return _rank_rows(s)[:k_final]


def per_example_metrics(
    row: dict[str, Any],
    *,
    method: str,
    score_fn: Callable[..., np.ndarray] | None = None,
    params: dict[str, Any] | None = None,
    oracle: bool = False,
    catalog_pop: bool = False,
) -> dict[str, Any] | None:
    """One val example: pool rerank, catalog pop, oracle, or raw retrieval -> ranking metrics."""
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    if not positives:
        return None

    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    retrieved_rows = np.asarray([app_to_row[a] for a in pool_apps], dtype=np.int64)

    if oracle:
        ranked = _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)[:K_FINAL]
    elif catalog_pop:
        ranked = popularity_catalog_ranked_indices(query_app_id=int(row["query_app_id"]), k_final=K_FINAL)
    elif score_fn is None:
        ranked = pool_scores_to_ranked_indices(pool_apps, np.asarray(ret_sc), k_final=K_FINAL)
    else:
        blend = score_fn(pool_apps, ret_sc, **(params or {}), pop_row=pop_row, app_to_row=app_to_row)
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)

    return {
        "method": method,
        "slice_name": row.get("slice_name", ""),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
        "OracleHit@K": hit_rate_at_k(
            _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)[:K_FINAL],
            positives,
            K_FINAL,
            app_ids,
        ),
        "OracleNDCG@K": ndcg_at_k(
            _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)[:K_FINAL],
            positives,
            K_FINAL,
            app_ids,
        ),
    }

## D2 — pointwise training

### Data (three different “val” concepts)

| Name | Source | Used for |
|------|--------|----------|
| **`train_pools`** | `train_ranker_v1/.../two_tower_v1.parquet` | Fit + inner tune only (disjoint from official val) |
| **`train_fit` / `train_tune`** | 90/10 split of `train_pools` (stratified by `slice_name`) | TF training + checkpoint selection |
| **`val_pools`** | `eval_offline_examples.jsonl` | **Report only** — head-to-head at the bottom; never passed to `train_pointwise_ranker` |

`train_tune` is **not** `val_pools`. It is a holdout slice of the **train ranker cohort** (~10% of examples → ~517k pair rows).

### What one training example is

Each frozen pool has up to 100 candidates. We expand to **one row per candidate**:

- Features: `retr_score`, `log_pop` (log1p train popularity)
- Label: `1` if app is in `validation_positive_app_ids_json`, else `0` (~2% positive rate)

The MLP predicts relevance per pair (pointwise), not a full list at once.

### How `train_pointwise_ranker` works (TensorFlow / Keras)

Implementation: `src/steam_review_ml/recommender/ranker_d2_pointwise.py`.

**Model:** small Keras MLP → sigmoid, loss = weighted **binary crossentropy** (`class_weight` upweights rare positives on `fit`).

**Why one epoch per loop (not `fit(..., epochs=20)` once)?**  
After each epoch we run extra eval that Keras does not do automatically:

1. **`train_loss`** — BCE from `model.fit` on `d2_fit_df`
2. **`tune_loss`** — same weighted BCE on `d2_tune_df` via `model.evaluate` + `sample_weight` (Keras `evaluate` does not accept `class_weight`; per-row weights match fit)
3. **`tune_NDCG@10`** — rerank each `train_tune` pool with the current model, mean NDCG (NumPy eval helpers, not TF loss)

**Early stopping** uses **`tune_NDCG` only** (patience 3): restore weights from the best NDCG epoch. BCE losses are for explainability (overfitting: train ↓, tune ↑).

NDCG is not differentiable, so it cannot be the Keras `loss`; we train on BCE and **select checkpoints** on the ranking metric we care about.

Training is slow mainly because step 3 runs ~5k pool `predict`s **per epoch**.

**Official ranking numbers** still come from the **val head-to-head** section below (12.5k `val_pools`), with fixed weights after training.

In [5]:
d2_fit_df = pointwise_rows_from_pools(train_fit, pop_row=pop_row, app_to_row=app_to_row)
d2_tune_df = pointwise_rows_from_pools(train_tune, pop_row=pop_row, app_to_row=app_to_row)
print(f"D2 pairs: fit={len(d2_fit_df):,} tune={len(d2_tune_df):,}  pos_rate={d2_fit_df['label'].mean():.4f}")

d2_model = None
d2_history = pd.DataFrame()

if TRAIN_D2:
    D2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    d2_cfg = PointwiseRankerConfig(epochs=20, early_stopping_patience=3, batch_size=4096)
    d2_model, d2_history = train_pointwise_ranker(
        fit_df=d2_fit_df,
        tune_df=d2_tune_df,
        tune_pools=train_tune,
        app_ids=app_ids,
        app_to_row=app_to_row,
        pop_row=pop_row,
        cfg=d2_cfg,
        k_final=K_FINAL,
        verbose=0,
    )
    d2_model.save(D2_MODEL_PATH)
    d2_history.to_csv(D2_OUTPUT_DIR / "train_history.csv", index=False)
    print(f"Saved D2 model to {D2_MODEL_PATH}")
    display(d2_history)
elif D2_MODEL_PATH.is_file():
    d2_model = load_pointwise_ranker(D2_MODEL_PATH)
    print(f"Loaded existing D2 model from {D2_MODEL_PATH}")
else:
    print("D2 not trained and no saved model found.")

D2 pairs: fit=4,652,200 tune=516,900  pos_rate=0.0219


2026-06-05 08:41:59.018388: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-05 08:41:59.068976: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780666919.078858    1196 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780666919.082522    1196 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780666919.131521    1196 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

D2 epoch 1/20: train_loss=1.2233 tune_loss=1.0964 tune_NDCG@10=0.1619
D2 epoch 2/20: train_loss=1.0991 tune_loss=1.0836 tune_NDCG@10=0.1626
D2 epoch 3/20: train_loss=1.0901 tune_loss=1.0793 tune_NDCG@10=0.1630
D2 epoch 4/20: train_loss=1.0849 tune_loss=1.0738 tune_NDCG@10=0.1640
D2 epoch 5/20: train_loss=1.0806 tune_loss=1.0701 tune_NDCG@10=0.1644
D2 epoch 6/20: train_loss=1.0759 tune_loss=1.0650 tune_NDCG@10=0.1646
D2 epoch 7/20: train_loss=1.0713 tune_loss=1.0607 tune_NDCG@10=0.1651
D2 epoch 8/20: train_loss=1.0690 tune_loss=1.0581 tune_NDCG@10=0.1652
D2 epoch 9/20: train_loss=1.0654 tune_loss=1.0555 tune_NDCG@10=0.1655
D2 epoch 10/20: train_loss=1.0634 tune_loss=1.0533 tune_NDCG@10=0.1663
D2 epoch 11/20: train_loss=1.0617 tune_loss=1.0520 tune_NDCG@10=0.1666
D2 epoch 12/20: train_loss=1.0605 tune_loss=1.0513 tune_NDCG@10=0.1666
D2 epoch 13/20: train_loss=1.0604 tune_loss=1.0511 tune_NDCG@10=0.1665
D2 epoch 14/20: train_loss=1.0593 tune_loss=1.0496 tune_NDCG@10=0.1677
D2 epoch 15/20:

,epoch,train_loss,tune_loss,tune_NDCG
0,1,1.223338,1.096366,0.161914
1,2,1.099122,1.083557,0.162557
2,3,1.090104,1.079317,0.163028
3,4,1.084881,1.073775,0.164048
4,5,1.080558,1.070134,0.164372
5,6,1.075904,1.064951,0.164579
6,7,1.071304,1.060668,0.165087
7,8,1.068990,1.058136,0.165170
8,9,1.065429,1.055509,0.165459
9,10,1.063438,1.053349,0.166315


## D3 — listwise training (ListNet)

**Not pairwise** — one row per example (~46k lists vs D2’s ~4.6M pair rows).

- Features per candidate: same as D2 (`retr_score`, `log_pop`).
- **ListNet loss:** softmax over pool scores vs uniform target over within-pool positives.
- Training loop mirrors D2: `train_fit` lists → fit; `train_tune` → tune loss + **tune NDCG** early stop.

Implementation: `src/steam_review_ml/recommender/ranker_d3_listwise.py`.

In [6]:
d3_fit_df = listwise_rows_from_pools(train_fit, pop_row=pop_row, app_to_row=app_to_row)
d3_tune_df = listwise_rows_from_pools(train_tune, pop_row=pop_row, app_to_row=app_to_row)
print(f"D3 lists: fit={len(d3_fit_df):,} tune={len(d3_tune_df):,}")

d3_model = None
d3_history = pd.DataFrame()

if TRAIN_D3:
    D3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    d3_cfg = ListwiseRankerConfig(epochs=20, early_stopping_patience=3, list_batch_size=256)
    d3_model, d3_history = train_listwise_ranker(
        fit_df=d3_fit_df,
        tune_df=d3_tune_df,
        tune_pools=train_tune,
        app_ids=app_ids,
        app_to_row=app_to_row,
        pop_row=pop_row,
        cfg=d3_cfg,
        k_final=K_FINAL,
        verbose=0,
    )
    d3_model.save(D3_MODEL_PATH)
    d3_history.to_csv(D3_OUTPUT_DIR / "train_history.csv", index=False)
    print(f"Saved D3 model to {D3_MODEL_PATH}")
    display(d3_history)
elif D3_MODEL_PATH.is_file():
    d3_model = load_listwise_ranker(D3_MODEL_PATH)
    print(f"Loaded existing D3 model from {D3_MODEL_PATH}")
else:
    print("D3 not trained and no saved model found.")

D3 lists: fit=46,522 tune=5,169
D3 epoch 1/20: train_loss=4.6411 tune_loss=4.1682 tune_NDCG@10=0.1644
D3 epoch 2/20: train_loss=4.3453 tune_loss=4.1063 tune_NDCG@10=0.1644
D3 epoch 3/20: train_loss=4.2617 tune_loss=4.0820 tune_NDCG@10=0.1642
D3 epoch 4/20: train_loss=4.2272 tune_loss=4.0721 tune_NDCG@10=0.1640
D3 epoch 5/20: train_loss=4.2005 tune_loss=4.0607 tune_NDCG@10=0.1637
Saved D3 model to /home/ryanr/workspace/steam_recommendations/artifacts/recs/rankers/d3_listwise_v1/listwise_ranker.keras


,epoch,train_loss,tune_loss,tune_NDCG
0,1,4.641062,4.168190,0.164371
1,2,4.345332,4.106274,0.164381
2,3,4.261657,4.082049,0.164209
3,4,4.227227,4.072115,0.164008
4,5,4.200514,4.060742,0.163674


## Val head-to-head (external val, fixed params)

Always includes: `two_tower_v1`, D1 logpop, `popularity_train`, `two_tower_v1_oracle`. Adds D2/D3 when model paths are set.

In [7]:
D1_SPEC = pool_rerank_registry()[METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND]


def score_d1_logpop(pool_apps, retrieval_scores, **_ignored):
    """Promoted D1 logpop blend (fixed alpha from registry)."""
    return rerank_scores_on_pool(
        pool_apps,
        retrieval_scores,
        D1_SPEC,
        pop_row=pop_row,
        app_to_row=app_to_row,
    )


HEAD_TO_HEAD: list[dict[str, Any]] = [
    {"method": POOL_METHOD, "kind": "pool_retrieval"},
    {"method": METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND, "kind": "pool_rerank", "score_fn": score_d1_logpop},
    {"method": "popularity_train", "kind": "catalog_pop"},
    {"method": f"{POOL_METHOD}_oracle", "kind": "oracle"},
]

if d2_model is not None:
    score_d2 = make_d2_pool_score_fn(d2_model, pop_row=pop_row, app_to_row=app_to_row)
    HEAD_TO_HEAD.insert(
        2,
        {
            "method": METHOD_TWO_TOWER_V1_RANKER_D2_POINTWISE_V1,
            "kind": "pool_rerank",
            "score_fn": score_d2,
        },
    )

if d3_model is not None:
    score_d3 = make_d3_pool_score_fn(d3_model, pop_row=pop_row, app_to_row=app_to_row)
    HEAD_TO_HEAD.insert(
        3,
        {
            "method": METHOD_TWO_TOWER_V1_RANKER_D3_LISTWISE_V1,
            "kind": "pool_rerank",
            "score_fn": score_d3,
        },
    )

val_rows: list[dict[str, Any]] = []
for row in val_pools:
    for spec in HEAD_TO_HEAD:
        kind = spec["kind"]
        if kind == "oracle":
            m = per_example_metrics(row, method=spec["method"], oracle=True)
        elif kind == "catalog_pop":
            m = per_example_metrics(row, method=spec["method"], catalog_pop=True)
        elif kind == "pool_retrieval":
            m = per_example_metrics(row, method=spec["method"], score_fn=None)
        elif kind == "pool_rerank":
            m = per_example_metrics(row, method=spec["method"], score_fn=spec["score_fn"])
        else:
            continue
        if m is not None:
            val_rows.append(m)

df_val = pd.DataFrame(val_rows)
print(f"methods in face-off: {sorted(df_val['method'].unique())}")

methods in face-off: ['popularity_train', 'two_tower_v1', 'two_tower_v1_heuristic_logpop_blend', 'two_tower_v1_oracle', 'two_tower_v1_ranker_d2_pointwise_v1', 'two_tower_v1_ranker_d3_listwise_v1']


In [8]:
display(Markdown("### Val ranking overall"))
display(
    df_val.groupby("method")[["Hit@K", "MAP@K", "NDCG@K", "MRR", "OracleHit@K", "OracleNDCG@K"]]
    .mean()
    .sort_values("NDCG@K", ascending=False)
)

display(Markdown("### Val ranking by slice"))
display(
    df_val.groupby(["slice_name", "method"])[["Hit@K", "NDCG@K", "MRR"]]
    .mean()
    .sort_values(["slice_name", "NDCG@K"], ascending=[True, False])
)

### Val ranking overall

,Hit@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K
method,,,,,,
two_tower_v1_oracle,0.51224,0.494053,0.498310,0.512240,0.51224,0.49831
two_tower_v1_heuristic_logpop_blend,0.19328,0.064321,0.092892,0.067059,0.51224,0.49831
two_tower_v1_ranker_d2_pointwise_v1,0.19008,0.059871,0.088751,0.062318,0.51224,0.49831
two_tower_v1_ranker_d3_listwise_v1,0.18168,0.057606,0.084934,0.059933,0.51224,0.49831
popularity_train,0.15112,0.050594,0.073109,0.052221,0.51224,0.49831
two_tower_v1,0.04680,0.010325,0.018161,0.011008,0.51224,0.49831


### Val ranking by slice

Hit@K    NDCG@K  \
slice_name            method                                                    
slice_a_multi_target  two_tower_v1_oracle                  0.773793  0.533618   
                      two_tower_v1_heuristic_logpop_blend  0.271724  0.068322   
                      two_tower_v1_ranker_d2_pointwise_v1  0.259310  0.062500   
                      two_tower_v1_ranker_d3_listwise_v1   0.256552  0.058748   
                      popularity_train                     0.128276  0.035444   
                      two_tower_v1                         0.091034  0.020537   
slice_b_single_target two_tower_v1_oracle                  0.496136  0.496136   
                      two_tower_v1_heuristic_logpop_blend  0.188450  0.094404   
                      two_tower_v1_ranker_d2_pointwise_v1  0.185817  0.090367   
                      two_tower_v1_ranker_d3_listwise_v1   0.177070  0.086547   
                      popularity_train                     0.152527  0.075428   
                      two_tower_v1                         0.044076  0.018015   

                                                                MRR  
slice_name            method                                         
slice_a_multi_target  two_tower_v1_oracle                  0.773793  
                      two_tower_v1_heuristic_logpop_blend  0.081396  
                      two_tower_v1_ranker_d2_pointwise_v1  0.071857  
                      two_tower_v1_ranker_d3_listwise_v1   0.067093  
                      popularity_train                     0.047469  
                      two_tower_v1                         0.021192  
slice_b_single_target two_tower_v1_oracle                  0.496136  
                      two_tower_v1_heuristic_logpop_blend  0.066176  
                      two_tower_v1_ranker_d2_pointwise_v1  0.061730  
                      two_tower_v1_ranker_d3_listwise_v1   0.059492  
                      popularity_train                     0.052514  
                      two_tower_v1                         0.010381

### Takeaway (this run)

**D1 `two_tower_v1_heuristic_logpop_blend` still wins** on val NDCG@10 overall and by slice. D2/D3 beat bare `two_tower_v1` but sit slightly below D1.

**Not a final verdict on learned rankers** — both D2 and D3 used a single small MLP (16 hidden units, default LR/epochs) with no hyperparameter search. Likely next steps if we revisit:

- Tune LR, batch size, epochs/patience, and hidden width on **`train_tune` only** (same discipline as D1 `alpha` on train pools).
- **Add another Dense layer** (or slightly wider MLP) — current capacity may be too shallow to beat a hand-tuned logpop blend.
- D1 itself may have room: **`alpha` was fixed at 0.2** from train-pool tuning in recs_013; a fresh grid on current train pools could move the D1 baseline.

Promote a learned ranker to the eval job only after it **beats tuned D1 on external val**, not just two-tower retrieval order.